In [ ]:
%pip install yfinance --quiet

# 03 · Ingesta de cotizaciones — APIs externas → Bronze

Extrae cotizaciones diarias de cada fuente del registro (`QUOTE_SOURCES`) para los símbolos operados, y las carga en una tabla Bronze **por fuente**, con sus campos nativos.

```
silver.transacciones ─▶ pedidos ─┬─ data912 ─▶ landing/.../cotizaciones/data912/*.jsonl ─Auto Loader─▶ bronze.data912_cotizaciones
                                 └─ yahoo   ─▶ landing/.../cotizaciones/yahoo/*.jsonl   ─Auto Loader─▶ bronze.yahoo_cotizaciones
                                    errores por símbolo de todas las fuentes ──────────────────────────▶ bronze.errores_api
```

| | |
|---|---|
| **Fuentes** | data912 (renta fija, acciones y CEDEARs en ARS) y Yahoo Finance (acciones y CEDEARs, incluidas variantes MEP) — DT-017 |
| **Errores** | Ningún símbolo corta la corrida: los errores quedan en `bronze.errores_api` (DT-018) |
| **Re-ejecución** | Si una fuente ya se extrajo hoy, no se vuelve a llamar a su API (salvo `force_extract`). La carga a Bronze es idempotente por archivo |

**Prerrequisito:** `setup/06_cotizaciones.sql` y el notebook `02_silver_transacciones`.

## Parámetros

In [ ]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.dropdown("force_extract", "false", ["true", "false"], "Forzar extracción")
CATALOG = dbutils.widgets.get("catalog")
FORCE_EXTRACT = dbutils.widgets.get("force_extract") == "true"

In [ ]:
import json
import os
import sys
from collections import Counter
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from pyspark.sql import functions as F

from iol_pipeline.bronze import STATUS_FAIL, ingest_files_to_bronze, validate_ingestion
from iol_pipeline.config import PipelineConfig
from iol_pipeline.cotizaciones import ERRORS_COLUMNS, build_requests, extract_source, extraction_window
from iol_pipeline.sources import QUOTE_SOURCES

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
silver_table = cfg.silver_table("transacciones")
errors_table = cfg.bronze_table("errores_api")

for source in QUOTE_SOURCES:
    print(f"{source.name:8} landing: {cfg.landing_path('cotizaciones/' + source.name)}")
    print(f"{'':8} bronze:  {cfg.bronze_table(source.bronze_table)}")
print(f"errores  {errors_table}")

## Precondiciones

In [ ]:
assert spark.catalog.tableExists(silver_table), f"No existe {silver_table}. Ejecutar 02_silver_transacciones"
for table in [cfg.bronze_table(s.bronze_table) for s in QUOTE_SOURCES] + [errors_table]:
    assert spark.catalog.tableExists(table), f"No existe {table}. Ejecutar setup/06_cotizaciones.sql"

silver = spark.table(silver_table)
assert silver.count() > 0, f"{silver_table} está vacía"

## Pedidos
Un pedido por símbolo operado. Cada fuente recibe solo los tipos de instrumento para los que tiene prioridad en el registro.

In [ ]:
requests = build_requests(silver)
by_type = Counter(r.tipo_instrumento for r in requests)
display(spark.createDataFrame(
    [(tipo, n, ", ".join(f"{s.name} (p{s.priority[tipo]})" for s in QUOTE_SOURCES if tipo in s.priority) or "sin fuente")
     for tipo, n in sorted(by_type.items(), key=lambda kv: -kv[1])],
    "tipo_instrumento STRING, simbolos LONG, fuentes STRING",
))

first_operation = silver.agg(F.min("fecha_operativa")).first()[0]
extracted_at = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
start, end = extraction_window(first_operation, extracted_at.date())
print(f"Ventana de extracción: {start} a {end} (hasta hoy, para conocer los splits posteriores)")

## Extracción a landing

In [ ]:
summaries, all_errors = [], []
for source in QUOTE_SOURCES:
    summary, errors = extract_source(
        source, requests, start, end,
        landing_dir=cfg.landing_path(f"cotizaciones/{source.name}"),
        extracted_at=extracted_at,
        force=FORCE_EXTRACT,
    )
    summaries.append(summary.to_dict())
    all_errors.extend(errors)

if all_errors:
    errors_schema = "simbolo STRING, ticker STRING, fuente STRING, error_type STRING, message STRING, _extracted_at TIMESTAMP"
    spark.createDataFrame([{c: e[c] for c in ERRORS_COLUMNS} for e in all_errors], errors_schema).writeTo(errors_table).append()

display(spark.createDataFrame(summaries))

### Errores de la extracción
Por fuente y tipo de error. `NOT_FOUND` es esperable: la fuente no tiene ese instrumento (ver cobertura en DT-017).

In [ ]:
if all_errors:
    display(
        spark.table(errors_table).where(F.col("_extracted_at") == F.lit(extracted_at))
        .groupBy("fuente", "error_type").agg(F.count("*").alias("simbolos"))
        .orderBy("fuente", F.desc("simbolos"))
    )
else:
    print("Sin errores nuevos (o extracciones omitidas)")

## Carga a Bronze
Auto Loader por fuente: solo procesa archivos de landing nuevos y valida que las filas escritas coincidan con las de los archivos.

In [ ]:
ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)
results = []
for source in QUOTE_SOURCES:
    table = cfg.bronze_table(source.bronze_table)
    ingest_files_to_bronze(
        spark,
        source_path=cfg.landing_path(f"cotizaciones/{source.name}"),
        target_table=table,
        checkpoint_path=cfg.checkpoint_path(f"cotizaciones_{source.name}"),
        columns=source.bronze_columns,
        ingested_at=ingested_at,
        file_format="json",
    )
    results.append(validate_ingestion(spark, table, ingested_at, file_format="json").to_dict())

display(spark.createDataFrame(results))

## Contenido de Bronze por fuente
Símbolos con datos y rango de fechas en la última extracción de cada fuente.

In [ ]:
pending_symbols = {s.name: len(s.requests_for(requests)) for s in QUOTE_SOURCES}
rows = []
for source in QUOTE_SOURCES:
    bronze = spark.table(cfg.bronze_table(source.bronze_table))
    last = bronze.agg(F.max("_extracted_at")).first()[0]
    stats = bronze.where(F.col("_extracted_at") == last).agg(
        F.countDistinct("simbolo").alias("simbolos"), F.count("*").alias("filas"),
        F.min("date").alias("desde"), F.max("date").alias("hasta"),
    ).first()
    rows.append((source.name, pending_symbols[source.name], stats["simbolos"], stats["filas"], last, stats["desde"], stats["hasta"]))

display(spark.createDataFrame(
    rows, "fuente STRING, simbolos_pedidos LONG, simbolos_con_datos LONG, filas LONG, extraccion TIMESTAMP, desde DATE, hasta DATE",
))

## Resultado
Se devuelve al orquestador como JSON. Los errores de la API no hacen fallar la corrida; solo falla si la carga a Bronze no es íntegra.

In [ ]:
failed = [r for r in results if r["status"] == STATUS_FAIL]
summary = {
    "status": STATUS_FAIL if failed else "OK",
    "message": "; ".join(r["message"] for r in failed) if failed else
               f"{sum(s['registros'] for s in summaries)} registros extraídos, {len(all_errors)} errores de API",
    "extracciones": summaries,
    "cargas": results,
}
print(json.dumps(summary, indent=2, default=str))
dbutils.notebook.exit(json.dumps(summary, default=str))